# Tutorial 2: Comparing MPC Nonlinearities — CuRL LUTs vs. SHAFT vs. CrypTen vs. Fission

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mugurbil/prix/blob/main/examples/02_approximations_and_ablations.ipynb)

One of **PriX**'s core strengths for MPC researchers is that it unifies four major nonlinear approximation paradigms under a single API:
- **`mode="lut"`** ([**CuRL, CAMLIS 2024**](https://eprint.iacr.org/2024/1045)): Haar & Biorthogonal-2.2 Wavelet Look-Up Tables + `[EGK+20]` probabilistic truncation.
- **`mode="shaft"`** ([**SHAFT, NDSS 2025**](https://eprint.iacr.org/2024/1965)): 1-round trigonometric-triple Fourier-series GeLU / SiLU / Erf / Tanh / Sigmoid + constant-round Forward-Euler ODE Softmax.
- **`mode="crypten"`** ([**CrypTen, NeurIPS 2021**](https://arxiv.org/abs/2109.00984)): Limit `exp`, Newton–Raphson `reciprocal` / `inv_sqrt`, Taylor `erf`, sigmoid-based `silu`.
- **`mode="fission"`** ([**Fission, IEEE CNS 2025**](https://eprint.iacr.org/2025/653)): Hybrid permute-and-reveal to a non-colluding plaintext evaluator party.

In this notebook, you will:
1. Run an accuracy and latency ablation across all 4 modes on `gelu`, `silu`, and `softmax`.
2. Customize fine-grained approximation hyperparameters via `prix.cfg` (e.g. Fourier terms, ODE steps, or LUT bit-widths).

In [ ]:
!pip install -q prix-jax

## 1. Accuracy & Latency Ablation Across All 4 Modes

With `prix.run(fn, mode=mode)`, switching between **CrypTen**, **SHAFT**, **CuRL LUTs**, and **Fission** requires changing only the `mode` string:

In [ ]:
import time
import jax
import jax.numpy as jnp
import numpy as np
import prix

def eval_nonlinearities():
    grid = jnp.linspace(-3.5, 3.5, 128, dtype=jnp.float32)
    x = prix.cryptensor(grid)

    # Warmup XLA compilation
    _ = x.gelu().get_plain_text()
    _ = x.silu().get_plain_text()
    _ = x.reshape(4, 32).softmax(dim=-1).get_plain_text()

    t0 = time.perf_counter()
    gelu_mpc = x.gelu().get_plain_text()
    t_gelu = (time.perf_counter() - t0) * 1e3

    t0 = time.perf_counter()
    silu_mpc = x.silu().get_plain_text()
    t_silu = (time.perf_counter() - t0) * 1e3

    t0 = time.perf_counter()
    sm_mpc = x.reshape(4, 32).softmax(dim=-1).get_plain_text()
    t_sm = (time.perf_counter() - t0) * 1e3

    gelu_ref = np.asarray(jax.nn.gelu(grid, approximate=False))
    silu_ref = np.asarray(jax.nn.silu(grid))
    sm_ref = np.asarray(jax.nn.softmax(grid.reshape(4, 32), axis=-1))

    return {
        "gelu_mae": float(np.mean(np.abs(np.asarray(gelu_mpc) - gelu_ref))),
        "gelu_ms": t_gelu,
        "silu_mae": float(np.mean(np.abs(np.asarray(silu_mpc) - silu_ref))),
        "silu_ms": t_silu,
        "softmax_mae": float(np.mean(np.abs(np.asarray(sm_mpc) - sm_ref))),
        "softmax_ms": t_sm,
    }

print(f"{'Mode':<10} | {'GeLU MAE':>10} {'(ms)':>8} | {'SiLU MAE':>10} {'(ms)':>8} | {'Softmax MAE':>11} {'(ms)':>8}")
print("-" * 76)
for mode in ["crypten", "shaft", "lut", "fission"]:
    m = prix.run(eval_nonlinearities, world_size=2, mode=mode)
    print(
        f"{mode:<10} | {m['gelu_mae']:10.5f} {m['gelu_ms']:8.2f} | "
        f"{m['silu_mae']:10.5f} {m['silu_ms']:8.2f} | "
        f"{m['softmax_mae']:11.5f} {m['softmax_ms']:8.2f}"
    )

## 2. Fine-Grained Hyperparameter Sweeps with `prix.cfg.temp_override`

Want to mix and match methods (e.g., use **SHAFT Fourier GeLU** while sweeping the number of Fourier series terms `fs_terms` from 4 to 16, or combine **SHAFT ODE Softmax** with **CuRL Wavelet LayerNorm**)? Use `prix.cfg.temp_override`:

In [ ]:
@prix.mpc.run_multiprocess(world_size=2, mode="shaft")
def sweep_fourier_terms():
    grid = jnp.linspace(-3.5, 3.5, 128, dtype=jnp.float32)
    ref = np.asarray(jax.nn.gelu(grid, approximate=False))
    x = prix.cryptensor(grid)

    out = {}
    for terms in [4, 6, 8, 12, 16]:
        with prix.cfg.temp_override({"functions.gelu.fs_terms": terms}):
            pred = np.asarray(x.gelu().get_plain_text())
            out[terms] = float(np.mean(np.abs(pred - ref)))
    return out

for terms, mae in sweep_fourier_terms().items():
    print(f"SHAFT Fourier GeLU (fs_terms={terms:2d}) -> MAE = {mae:.6f}")